## Importing libraries

In [1]:
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


## Loading dataset

In [5]:
df=pd.read_csv("Ferry Tickets.csv")

In [6]:
df

,_id,Timestamp,Redemption Count,Sales Count
0,1,2025-12-21T22:30:00,14,16
1,2,2025-12-21T22:15:00,1,0
2,3,2025-12-21T22:00:00,2,0
3,4,2025-12-21T21:30:00,11,1
4,5,2025-12-21T21:15:00,10,0
...,...,...,...,...
261533,261534,2015-05-04T16:00:00,0,2
261534,261535,2015-05-01T16:00:00,1,0
261535,261536,2015-05-01T15:45:00,0,1
261536,261537,2015-05-01T15:15:00,0,2


In [7]:
df.isnull().sum()

_id                 0
Timestamp           0
Redemption Count    0
Sales Count         0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.columns

Index(['_id', 'Timestamp', 'Redemption Count', 'Sales Count'], dtype='object')

In [10]:
df.describe()

,_id,Redemption Count,Sales Count
count,261538.000000,261538.000000,261538.000000
mean,130769.500000,48.885030,49.599106
std,75499.661689,104.549336,99.862285
min,1.000000,0.000000,0.000000
25%,65385.250000,3.000000,3.000000
50%,130769.500000,11.000000,13.000000
75%,196153.750000,40.000000,48.000000
max,261538.000000,7216.000000,7229.000000


In [11]:
df.columns

Index(['_id', 'Timestamp', 'Redemption Count', 'Sales Count'], dtype='object')

## Load and prepare

In [12]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Year"]      = df["Timestamp"].dt.year
df["Month"]     = df["Timestamp"].dt.month
df["Hour"]      = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek   # 0 = Monday

In [13]:
# Aggregations

annual = (df.groupby("Year")
            .agg(Sold=("Sales Count", "sum"), Redeemed=("Redemption Count", "sum"))
            .reset_index())
annual["Util%"] = (annual["Redeemed"] / annual["Sold"] * 100).round(1)

monthly_pivot = {}
for yr in [2023, 2024, 2025]:
    monthly_pivot[yr] = (df[df["Year"] == yr]
                           .groupby("Month")["Redemption Count"].sum()
                           .reindex(range(1, 13), fill_value=0).values)

hourly = df.groupby("Hour")["Redemption Count"].sum().values

dow = (df.groupby("DayOfWeek")["Redemption Count"].sum().values)

## Color pallete

In [14]:
BLUE   = "#378ADD"
GREEN  = "#1D9E75"
PURPLE = "#7F77DD"
CORAL  = "#D85A30"
GRAY   = "#B4B2A9"
LIGHT  = "#F1EFE8"

def fmt_tick(val, _):
    if val >= 1_000_000: return f"{val/1_000_000:.1f}M"
    if val >= 1_000:     return f"{val/1_000:.0f}K"
    return str(int(val))


## Canvas

In [15]:
fig = plt.figure(figsize=(16, 18), facecolor="white")
fig.suptitle("Ferry Tickets — Operations Dashboard  (2015–2025)",
             fontsize=16, fontweight="bold", y=0.98)

gs = fig.add_gridspec(4, 2, hspace=0.45, wspace=0.28,
                      left=0.07, right=0.96, top=0.94, bottom=0.04)


## Key performance inidicators

In [16]:
kpi_ax = fig.add_subplot(gs[0, :])
kpi_ax.axis("off")

kpis = [
    ("12.97 M", "Total tickets sold\n2015 – 2025"),
    ("12.79 M", "Total redeemed"),
    ("98.6 %",  "Utilization rate\n(redeemed / sold)"),
    ("~186 K",  "Unredeemed tickets\n(1.4 % unused)"),
]
for i, (val, lbl) in enumerate(kpis):
    x = 0.125 + i * 0.25
    kpi_ax.text(x, 0.72, val, ha="center", va="center",
                fontsize=20, fontweight="bold", color="#222")
    kpi_ax.text(x, 0.22, lbl, ha="center", va="center",
                fontsize=10, color="#666", linespacing=1.5)
    rect = plt.Rectangle((x - 0.115, 0.0), 0.23, 1.0,
                          transform=kpi_ax.transAxes,
                          color=LIGHT, zorder=-1, clip_on=False)
    kpi_ax.add_patch(rect)


## 1. Annual bar chart

In [17]:
ax1 = fig.add_subplot(gs[1, 0])
x  = np.arange(len(annual))
w  = 0.4
ax1.bar(x - w/2, annual["Sold"],     width=w, color=BLUE,  label="Sold",     zorder=2)
ax1.bar(x + w/2, annual["Redeemed"], width=w, color=GREEN, label="Redeemed", zorder=2)
ax1.set_xticks(x)
ax1.set_xticklabels(annual["Year"], fontsize=9, rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_tick))

ax1.set_title("Annual tickets sold vs redeemed", fontsize=11, pad=8)
ax1.legend(fontsize=9)
ax1.grid(axis="y", color="#e0e0e0", zorder=1)
ax1.set_axisbelow(True)


## 2. Monthly seasonal lines

In [18]:
ax2 = fig.add_subplot(gs[1, 1])
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
colors_yr = {2023: PURPLE, 2024: BLUE, 2025: GREEN}
for yr, col in colors_yr.items():
    ax2.plot(range(1, 13), monthly_pivot[yr], color=col,
             marker="o", markersize=4, linewidth=1.8, label=str(yr))
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_labels, fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_tick))
ax2.set_title("Monthly redemption pattern (2023–2025)", fontsize=11, pad=8)
ax2.legend(fontsize=9)
ax2.grid(color="#e0e0e0")
ax2.set_axisbelow(True)

## 3. Hourly bar chart

In [19]:
ax3 = fig.add_subplot(gs[2, 0])
bar_colors = [BLUE if 10 <= h <= 14 else GRAY for h in range(24)]
ax3.bar(range(24), hourly, color=bar_colors, zorder=2)
ax3.set_xticks(range(0, 24, 2))
ax3.set_xticklabels([f"{h}h" for h in range(0, 24, 2)], fontsize=9)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_tick))
ax3.set_title("Ridership by hour of day", fontsize=11, pad=8)
ax3.grid(axis="y", color="#e0e0e0", zorder=1)
ax3.set_axisbelow(True)

from matplotlib.patches import Patch
ax3.legend(handles=[Patch(color=BLUE, label="Peak 10am–2pm"),
                    Patch(color=GRAY, label="Off-peak")], fontsize=9)


## 4. Day of week bar chart

In [20]:
ax4 = fig.add_subplot(gs[2, 1])
day_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
dow_colors = [BLUE if d >= 5 else GRAY for d in range(7)]
ax4.bar(day_labels, dow, color=dow_colors, zorder=2)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_tick))
ax4.set_title("Ridership by day of week", fontsize=11, pad=8)
ax4.grid(axis="y", color="#e0e0e0", zorder=1)
ax4.set_axisbelow(True)
ax4.legend(handles=[Patch(color=BLUE, label="Weekend"),
                    Patch(color=GRAY, label="Weekday")], fontsize=9)


## 5. Utilisation rate line

In [21]:
ax5 = fig.add_subplot(gs[3, 0])
ax5.plot(annual["Year"], annual["Util%"], color=CORAL,
         marker="o", markersize=5, linewidth=2, zorder=3)
ax5.fill_between(annual["Year"], annual["Util%"], 100,
                 where=(annual["Util%"] >= 100),
                 alpha=0.15, color=GREEN, label="Above 100%")
ax5.fill_between(annual["Year"], annual["Util%"], 100,
                 where=(annual["Util%"] < 100),
                 alpha=0.15, color=CORAL, label="Below 100%")
ax5.axhline(100, color="#999", linewidth=1, linestyle="--")
ax5.set_ylim(75, 110)
ax5.set_xticks(annual["Year"])
ax5.set_xticklabels(annual["Year"], fontsize=9, rotation=45)
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax5.set_title("Utilization rate by year", fontsize=11, pad=8)
ax5.legend(fontsize=9)
ax5.grid(color="#e0e0e0")
ax5.set_axisbelow(True)


## 6. Key insights text pannel

In [22]:
ax6 = fig.add_subplot(gs[3, 1])
ax6.axis("off")
ax6.set_title("Key insights", fontsize=11, pad=8, loc="left", x=0.02)

insights = [
    ("Summer dominates:",  "Jun–Aug = ~62% of annual volume"),
    ("Weekend surge:",     "Sat + Sun = ~42% of weekly trips"),
    ("Peak window:",       "10am–2pm = 46% of daily redemptions"),
    ("Near-perfect util:", "Post-2017 consistently ≥ 100%"),
    ("2025 on track:",     "Highest-volume year on record"),
]
for i, (bold, rest) in enumerate(insights):
    y = 0.85 - i * 0.18
    ax6.text(0.03, y, bold, fontsize=10, fontweight="bold",
             color="#222", transform=ax6.transAxes, va="center")
    ax6.text(0.03, y - 0.07, rest, fontsize=9, color="#555",
             transform=ax6.transAxes, va="center")
    ax6.plot([0.02, 0.98], [y - 0.12, y - 0.12],
             color="#e0e0e0", linewidth=0.8,
             transform=ax6.transAxes, clip_on=False)

ax6.text(0.03, 0.02,
         "* Redemptions > Sales suggests multi-use passes or carry-over redemptions",
         fontsize=8, color="#999", transform=ax6.transAxes)


Text(0.03, 0.02, '* Redemptions > Sales suggests multi-use passes or carry-over redemptions')

## Save

In [23]:
plt.savefig("ferry_tickets_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard saved to ferry_tickets_dashboard.png")

Dashboard saved to ferry_tickets_dashboard.png


/var/folders/8x/2hxqp0f90kg0qx1pzkl7ckg80000gn/T/ipykernel_32141/2626775565.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analytical methodology

### 1. Data ingestion and structuring


In [24]:
# Convert Timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort data
df = df.sort_values('Timestamp')

# Set index (important for resampling)
df.set_index('Timestamp', inplace=True)

### Multi-resolution Aggregation

In [25]:
# 15-minute aggregation
df_15min = df.resample('15T').sum()

# Hourly aggregation
df_hourly = df.resample('H').sum()

# Daily aggregation
df_daily = df.resample('D').sum()

/var/folders/8x/2hxqp0f90kg0qx1pzkl7ckg80000gn/T/ipykernel_32141/832292405.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_15min = df.resample('15T').sum()
/var/folders/8x/2hxqp0f90kg0qx1pzkl7ckg80000gn/T/ipykernel_32141/832292405.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H').sum()


### 2. Data quality and consistency checks
Missing/ irregular intervals

In [26]:
# Create complete timeline
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15T')

# Reindex
df_15min = df_15min.reindex(full_range)

# Check missing
missing_intervals = df_15min[df_15min.isnull().any(axis=1)]
print("Missing intervals:", len(missing_intervals))

Missing intervals: 0


/var/folders/8x/2hxqp0f90kg0qx1pzkl7ckg80000gn/T/ipykernel_32141/4280915555.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15T')


### Negative/ Zero anomalies

In [27]:
# Negative values
negative_check = df[(df['Sales Count'] < 0) | (df['Redemption Count'] < 0)]

# Zero activity
zero_activity = df[(df['Sales Count'] == 0) & (df['Redemption Count'] == 0)]

### Smooth extreme python rolling

In [28]:
# Rolling mean smoothing (example: hourly)
df_hourly['Sales Smooth'] = df_hourly['Sales Count'].rolling(window=3, min_periods=1).mean()
df_hourly['Redemption Smooth'] = df_hourly['Redemption Count'].rolling(window=3, min_periods=1).mean()

### 3. Capacity Utilisation feature engineering